In [8]:
import os

# 设置 Hugging Face Mirror
os.environ['HF_ENDPOINT'] = 'https://hf-mirror.com'

In [17]:
import torch # type: ignore
from transformers import BertTokenizer, BertForMaskedLM
import numpy as np
import pandas as pd
from tqdm import tqdm
import json
import sys
from datetime import datetime

In [10]:
# 创建输出文件夹
output_dir = 'ancient_bert_embedding_data'
os.makedirs(output_dir, exist_ok=True)

# 创建日志文件
log_file = os.path.join(output_dir, 'extraction_log.txt')
def log_print(message, also_print=True):
    """同时写入日志文件和控制台的函数"""
    if also_print:
        print(message)
    with open(log_file, 'a', encoding='utf-8') as f:
        f.write(message + '\n')

In [22]:
# ==============================================================================
# 步骤一：从 本地 加载古代中文模型
# ==============================================================================

LOCAL_MODEL_PATH = './guwenbert_local' 
# 1. 加载分词器
tokenizer = BertTokenizer.from_pretrained(LOCAL_MODEL_PATH)
print("√ 分词器 (Tokenizer) 加载成功")
        
# 2. 加载模型
model = BertForMaskedLM.from_pretrained(LOCAL_MODEL_PATH)
print("√ 模型 (Model) 加载成功")
        
# 3. 放到 GPU 上 (如果可用)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)
model.eval()
print(f"√ 模型已移动到设备: {device}")

# 动态获取模型层数信息
num_layers = model.config.num_hidden_layers
hidden_size = model.config.hidden_size

You are using a model of type roberta to instantiate a model of type bert. This is not supported for all configurations of models and can yield errors.


√ 分词器 (Tokenizer) 加载成功


Some weights of BertForMaskedLM were not initialized from the model checkpoint at ./guwenbert_local and are newly initialized: ['bert.embeddings.LayerNorm.bias', 'bert.embeddings.LayerNorm.weight', 'bert.embeddings.position_embeddings.weight', 'bert.embeddings.token_type_embeddings.weight', 'bert.embeddings.word_embeddings.weight', 'bert.encoder.layer.0.attention.output.LayerNorm.bias', 'bert.encoder.layer.0.attention.output.LayerNorm.weight', 'bert.encoder.layer.0.attention.output.dense.bias', 'bert.encoder.layer.0.attention.output.dense.weight', 'bert.encoder.layer.0.attention.self.key.bias', 'bert.encoder.layer.0.attention.self.key.weight', 'bert.encoder.layer.0.attention.self.query.bias', 'bert.encoder.layer.0.attention.self.query.weight', 'bert.encoder.layer.0.attention.self.value.bias', 'bert.encoder.layer.0.attention.self.value.weight', 'bert.encoder.layer.0.intermediate.dense.bias', 'bert.encoder.layer.0.intermediate.dense.weight', 'bert.encoder.layer.0.output.LayerNorm.bias', 

√ 模型 (Model) 加载成功
√ 模型已移动到设备: cpu


In [23]:
# ==============================================================================
# 步骤二：加载古代关系描述数据
# ==============================================================================

description_file = r"D:\\HuaweiMoveData\\Users\\黄奕\\Desktop\\Archive of OSF Storage\\my\\plms\\ancient_descriptions.json"
log_print(f"\n正在加载古代关系描述文件: {description_file}")

try:
    with open(description_file, 'r', encoding='utf-8') as f:
        descriptions = json.load(f)
    
    if not descriptions or not isinstance(descriptions, dict):
        log_print(f"✗ 致命错误：文件格式错误")
        log_print("期望格式: {\"关系名\": \"古代描述文本\", ...}")
        sys.exit(1)
    
    log_print(f"✓ 成功加载 {len(descriptions)} 条古代关系描述")
    
    # 显示前3个示例
    log_print("\n古代关系数据格式验证（前3个示例）:")
    for i, (rel, desc) in enumerate(list(descriptions.items())[:3]):
        preview = desc[:40] + "..." if len(desc) > 40 else desc
        log_print(f"  {i+1}. {rel}: {preview}")
    
except Exception as e:
    log_print(f"✗ 致命错误：读取古代关系文件失败 - {e}")
    sys.exit(1)



正在加载古代关系描述文件: D:\\HuaweiMoveData\\Users\\黄奕\\Desktop\\Archive of OSF Storage\\my\\plms\\ancient_descriptions.json
✓ 成功加载 160 条古代关系描述

古代关系数据格式验证（前3个示例）:
  1. 夫-妻: 夫者，天也，妻者，地也。天尊地卑，乾坤定矣。夫主外，以成事立业，立身扬名；妻主内...
  2. 父-子: 父者，子之天也。严而有威，慈而含章，立身以教，垂范以导。子者，父之嗣也，承其血脉...
  3. 父-女: 父女之伦，本乎天性。父者，严慈并济，为家之栋梁，教女以德容言功，导其明礼义、守闺...


In [24]:
# ==============================================================================
# 步骤三：配置古代语境的提示语模板
# ==============================================================================

prompt_template = "此{relation}之最显特征乃[MASK]。"
log_print(f"\n古代语境提示语模板: \"{prompt_template}\"")
log_print("注意：使用古代汉语风格的提示语，适配古文BERT模型")


古代语境提示语模板: "此{relation}之最显特征乃[MASK]。"
注意：使用古代汉语风格的提示语，适配古文BERT模型


In [25]:
# ==============================================================================
# 步骤四：核心功能 - 提取所有层的嵌入向量
# ==============================================================================

log_print("\n" + "="*80)
log_print(f"开始提取古代语境嵌入向量（CLS + MASK + 关系名，共{num_layers}层）...")
log_print("="*80)

# 初始化存储结构
cls_embeddings = {f'layer{i}': [] for i in range(num_layers)}
mask_embeddings = {f'layer{i}': [] for i in range(num_layers)}
mask_embeddings['output'] = []

relation_embeddings = {f'layer{i}': [] for i in range(num_layers)}
relation_embeddings['output'] = []

relation_names = []
skipped_relations = []
error_relations = []

with torch.no_grad():
    for rel, desc in tqdm(descriptions.items(), desc="处理古代关系", ncols=100):
        try:
            # 构造古代语境的输入文本
            desc_clean = desc.rstrip('。！？,.!?;；')
            text = f"{desc_clean}。{prompt_template.format(relation=rel)}"
            
            # 分词和编码
            inputs = tokenizer(text, return_tensors='pt', max_length=512, truncation=True, padding=False)
            inputs = {key: val.to(device) for key, val in inputs.items()}
            
            # 前向传播，获取所有层的隐藏状态
            outputs = model(**inputs, output_hidden_states=True)
            
            # 找到各种token的位置
            input_ids = inputs['input_ids'][0]
            cls_index = 0  # CLS token 总是在位置0
            
            # 找到 MASK token 的位置
            mask_indices = torch.where(input_ids == tokenizer.mask_token_id)[0]
            if len(mask_indices) == 0:
                skipped_relations.append(rel)
                log_print(f"  ⚠ 跳过 '{rel}'：未找到 [MASK] token", also_print=False)
                continue
            mask_index = mask_indices[0].item()
            
            # 找到关系名token的位置
            relation_tokens = tokenizer.encode(rel, add_special_tokens=False)
            relation_index = None
            if len(relation_tokens) > 0:
                relation_token_id = relation_tokens[0]
                relation_token_positions = torch.where(input_ids == relation_token_id)[0]
                if len(relation_token_positions) > 0:
                    relation_index = relation_token_positions[0].item()
            
            # 提取所有层的嵌入向量
            # outputs.hidden_states[0] 是 embedding 层
            # outputs.hidden_states[1] 到 outputs.hidden_states[12] 是 encoder 的12层
            all_hidden_states = outputs.hidden_states

            for layer_idx in range(num_layers):
                # encoder layer 0 对应 hidden_states[1]，以此类推
                hidden_state = all_hidden_states[layer_idx + 1] 
                
                # 提取 CLS token 嵌入
                cls_emb = hidden_state[0, cls_index, :].cpu().numpy()
                cls_embeddings[f'layer{layer_idx}'].append(cls_emb)
                
                # 提取 MASK token 嵌入
                mask_emb = hidden_state[0, mask_index, :].cpu().numpy()
                mask_embeddings[f'layer{layer_idx}'].append(mask_emb)
                
                # 提取关系名token嵌入
                if relation_index is not None:
                    rel_emb = hidden_state[0, relation_index, :].cpu().numpy()
                    relation_embeddings[f'layer{layer_idx}'].append(rel_emb)
                else:
                    # 如果找不到关系名token，用零向量填充
                    relation_embeddings[f'layer{layer_idx}'].append(np.zeros(hidden_size))
            
            # 提取最终输出层的嵌入（用于 embedding_output.csv）
            final_hidden_state = all_hidden_states[-1]  # 最后一层
            
            mask_output = final_hidden_state[0, mask_index, :].cpu().numpy()
            mask_embeddings['output'].append(mask_output)
            
            if relation_index is not None:
                rel_output = final_hidden_state[0, relation_index, :].cpu().numpy()
                relation_embeddings['output'].append(rel_output)
            else:
                relation_embeddings['output'].append(np.zeros(hidden_size))
            
            relation_names.append(rel)
            
        except Exception as e:
            error_relations.append((rel, str(e)))
            log_print(f"  ✗ 错误处理 '{rel}': {e}", also_print=False)
            continue


开始提取古代语境嵌入向量（CLS + MASK + 关系名，共12层）...


处理古代关系: 100%|███████████████████████████████████████████████| 160/160 [00:17<00:00,  9.32it/s]


In [26]:
# ==============================================================================
# 步骤五：保存为CSV文件
# ==============================================================================


try:
    # 列名使用数字索引（0, 1, 2, ..., hidden_size-1）
    column_names = list(range(hidden_size))
    
    # 保存 CLS token 的嵌入向量
    log_print("\n保存古代语境CLS token的嵌入向量...")
    for layer_idx in range(num_layers):
        df = pd.DataFrame(cls_embeddings[f'layer{layer_idx}'], index=relation_names, columns=column_names)
        df.index.name = 'word'
        filename = os.path.join(output_dir, f'CLS_encoder_layer{layer_idx}.csv')
        df.to_csv(filename, encoding='utf-8-sig')
        log_print(f"  ✓ {filename}")
    
    # 保存 MASK token 的嵌入向量
    log_print("\n保存古代语境MASK token的嵌入向量...")
    for layer_idx in range(num_layers):
        df = pd.DataFrame(mask_embeddings[f'layer{layer_idx}'], index=relation_names, columns=column_names)
        df.index.name = 'word'
        filename = os.path.join(output_dir, f'MASK_encoder_layer{layer_idx}.csv')
        df.to_csv(filename, encoding='utf-8-sig')
        log_print(f"  ✓ {filename}")
    
    # 保存 MASK token 的最终输出
    df = pd.DataFrame(mask_embeddings['output'], index=relation_names, columns=column_names)
    df.index.name = 'word'
    filename = os.path.join(output_dir, 'MASK_embedding_output.csv')
    df.to_csv(filename, encoding='utf-8-sig')
    log_print(f"  ✓ {filename}")
    
    # 保存关系名token的嵌入向量
    log_print("\n保存古代语境关系名token的嵌入向量...")
    for layer_idx in range(num_layers):
        df = pd.DataFrame(relation_embeddings[f'layer{layer_idx}'], index=relation_names, columns=column_names)
        df.index.name = 'word'
        filename = os.path.join(output_dir, f'关系_encoder_layer{layer_idx}.csv')
        df.to_csv(filename, encoding='utf-8-sig')
        log_print(f"  ✓ {filename}")
    
    # 保存关系名token的最终输出
    df = pd.DataFrame(relation_embeddings['output'], index=relation_names, columns=column_names)
    df.index.name = 'word'
    filename = os.path.join(output_dir, '关系_embedding_output.csv')
    df.to_csv(filename, encoding='utf-8-sig')
    log_print(f"  ✓ {filename}")
    
except Exception as e:
    log_print(f"✗ 保存古代语境文件时出错: {e}")
    sys.exit(1)


保存古代语境CLS token的嵌入向量...
  ✓ ancient_bert_embedding_data\CLS_encoder_layer0.csv
  ✓ ancient_bert_embedding_data\CLS_encoder_layer1.csv
  ✓ ancient_bert_embedding_data\CLS_encoder_layer2.csv
  ✓ ancient_bert_embedding_data\CLS_encoder_layer3.csv
  ✓ ancient_bert_embedding_data\CLS_encoder_layer4.csv
  ✓ ancient_bert_embedding_data\CLS_encoder_layer5.csv
  ✓ ancient_bert_embedding_data\CLS_encoder_layer6.csv
  ✓ ancient_bert_embedding_data\CLS_encoder_layer7.csv
  ✓ ancient_bert_embedding_data\CLS_encoder_layer8.csv
  ✓ ancient_bert_embedding_data\CLS_encoder_layer9.csv
  ✓ ancient_bert_embedding_data\CLS_encoder_layer10.csv
  ✓ ancient_bert_embedding_data\CLS_encoder_layer11.csv

保存古代语境MASK token的嵌入向量...
  ✓ ancient_bert_embedding_data\MASK_encoder_layer0.csv
  ✓ ancient_bert_embedding_data\MASK_encoder_layer1.csv
  ✓ ancient_bert_embedding_data\MASK_encoder_layer2.csv
  ✓ ancient_bert_embedding_data\MASK_encoder_layer3.csv
  ✓ ancient_bert_embedding_data\MASK_encoder_layer4.csv
  ✓ anc